# Reconstruction Impact: Region and Sensitivity

This notebook defines reconstruction-driven high-diversity **Region** using reconstructed `Neff`, quantifies its absolute area, and evaluates threshold and spatial-scale sensitivity. The Level1 anatomy map is shown separately for downstream region definition.

**Evidence boundary.** Both maps are reproducible window assignments. v0.1 does not impose connected components and does not infer tissue mechanism.

In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Could not locate the REVISE repository root.')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from revise.analysis.basic.spatial_region import assign_square_windows, compute_window_diversity, summarize_region
from revise.analysis.reconstruction_impact import load_reconstruction_impact_config

CONFIG_PATH = Path(os.environ.get(
    'REVISE_RECONSTRUCTION_IMPACT_CONFIG',
    REPO_ROOT / 'configs/analysis/reconstruction_impact_visiumhd_p1crc.yaml',
))
config = load_reconstruction_impact_config(CONFIG_PATH)
output_root = Path(os.environ.get('REVISE_ANALYSIS_OUTPUT_ROOT', REPO_ROOT / config['output']['dir']))
spatial_dir = output_root / 'spatial'
unit_window_map = pd.read_csv(spatial_dir / 'unit_window_map.csv.gz', index_col='unit_id')
anatomy_window_map = pd.read_csv(spatial_dir / 'anatomy_window_map.csv.gz', index_col='unit_id')
anatomy_region_map = pd.read_csv(spatial_dir / 'anatomy_region_map.csv.gz')
coordinates = unit_window_map[['x', 'y']]
raw_labels = unit_window_map['raw_cluster']
recon_labels = unit_window_map['recon_cluster']

In [ ]:
grid_origin = (
    float(unit_window_map['grid_origin_x'].iloc[0]),
    float(unit_window_map['grid_origin_y'].iloc[0]),
)
main_side = config['spatial_region']['window_side_length_um']
main_windows = assign_square_windows(coordinates, window_side_length=main_side, origin=grid_origin)
main_metrics = compute_window_diversity(
    main_windows, raw_labels, recon_labels,
    min_units_per_window=config['spatial_region']['min_units_per_window'],
)
main_summary, main_region = summarize_region(
    main_metrics, threshold=config['spatial_region']['neff_threshold'], window_side_length=main_side,
)
main_summary

In [ ]:
threshold_rows = []
for threshold in config['spatial_region']['neff_thresholds']:
    summary, _ = summarize_region(main_metrics, threshold=threshold, window_side_length=main_side)
    threshold_rows.append(summary.assign(scale_kind='threshold'))
for side_length in config['spatial_region']['window_side_lengths_um']:
    windows = assign_square_windows(coordinates, window_side_length=side_length, origin=grid_origin)
    metrics = compute_window_diversity(
        windows, raw_labels, recon_labels,
        min_units_per_window=config['spatial_region']['min_units_per_window'],
    )
    summary, _ = summarize_region(
        metrics, threshold=config['spatial_region']['neff_threshold'], window_side_length=side_length,
    )
    threshold_rows.append(summary.assign(scale_kind='window_side'))
sensitivity = pd.concat(threshold_rows, ignore_index=True)
sensitivity.to_csv(spatial_dir / 'region_sensitivity.csv', index=False)
main_region.to_csv(spatial_dir / 'region_summary_windows.csv.gz', index=False, compression='gzip')
main_summary.to_csv(spatial_dir / 'region_summary.csv', index=False)
sensitivity

In [ ]:
figure_dir = spatial_dir / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)
centers = main_windows.groupby('window_id')[['x', 'y']].mean().reset_index()
plot_data = centers.merge(main_region, on='window_id')
fig, axes = plt.subplots(1, 4, figsize=(18, 4), constrained_layout=True)
for axis, column, title, cmap in zip(
    axes[:3], ['neff_raw', 'neff_recon', 'delta_neff'],
    ['Raw Neff', 'Reconstructed Neff', 'Delta Neff'], ['viridis', 'viridis', 'coolwarm'],
):
    values = plot_data[column]
    kwargs = {'cmap': cmap}
    if column == 'delta_neff':
        limit = np.nanmax(np.abs(values))
        kwargs.update(vmin=-limit, vmax=limit)
    image = axis.scatter(plot_data['x'], plot_data['y'], c=values, s=12, **kwargs)
    axis.set_title(title)
    axis.set_aspect('equal')
    fig.colorbar(image, ax=axis)
in_region = plot_data['in_region']
axes[3].scatter(plot_data['x'], plot_data['y'], c='#d9d9d9', s=12)
axes[3].scatter(plot_data.loc[in_region, 'x'], plot_data.loc[in_region, 'y'], c='#b2182b', s=12)
axes[3].set_title(f"High-diversity Region: Neff >= {config['spatial_region']['neff_threshold']}")
axes[3].set_aspect('equal')
fig.savefig(figure_dir / 'region_panels.png', dpi=180)
plt.show()

anatomy_centers = anatomy_window_map.groupby('window_id')[['x', 'y']].mean().reset_index()
anatomy_plot = anatomy_centers.merge(anatomy_region_map[['window_id', 'region_label']], on='window_id')
colors = {'Tumor': '#bcbd22', 'Normal': '#4c78a8', 'Interface': '#e45756', 'Other': '#d9d9d9'}
fig, axis = plt.subplots(figsize=(5, 4), constrained_layout=True)
axis.scatter(anatomy_plot['x'], anatomy_plot['y'], c=anatomy_plot['region_label'].map(colors), s=12)
axis.set_title('Level1 anatomy regions')
axis.set_aspect('equal')
fig.savefig(figure_dir / 'anatomy_region_map.png', dpi=180)
plt.show()

threshold_curve = sensitivity.loc[sensitivity['scale_kind'] == 'threshold']
ax = threshold_curve.plot(x='threshold', y='region_area', marker='o', legend=False)
ax.set_ylabel('Region area (um2)')
ax.figure.tight_layout()
ax.figure.savefig(figure_dir / 'threshold_area_curve.png', dpi=180)
plt.show()